# 02 · Model config, and prediction on a test set

Trains on the labelled data from one cleaning run, less a test set (a file of IDs like
`to_predict.txt`) and any blacklisted IDs, then scores the test set. The configuration
is saved as `config/model_<stamp>.json` for notebook 04 to train from.

For the NN, a validation slice is carved from the training rows for early stopping and
checkpointing; the test set is never used for either.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))      # repo root, so `import classifier` works

DATA_DIR, CONFIG_DIR, OUTPUT_DIR = "../data", "../config", "../outputs"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from classifier import io, evaluation, plotting
from classifier.features import VectorizerSettings
from classifier.nb import NBSettings
from classifier.model import CategoryClassifier, ModelConfig, validation_split, hierarchy_lookup

STAMP = io.stamp()
print("run stamp:", STAMP)

## Parameters

In [ ]:
# ---- data ------------------------------------------------------------------------------
CLEANING_STAMP = "YYYYMMDD_HHMMSS"   # from notebook 01
TEST_SET_FILE = "to_predict.txt"   # in data/: IDs to hold out and score. None = random split
TEST_FRAC = 0.1                    # used only when TEST_SET_FILE is None
BLACKLIST_FILE = None              # in data/: IDs removed from everything. None = off
NN_VAL_FRAC = 0.05                 # NN only: validation slice of the training rows
SEED = 42

# ---- which model --------------------------------------------------------------
MODEL_TYPE = "nb"                  # "nb" (benchmark) | "nn"

# ---- rare-token pruning, counted on training rows only --------------------------
MIN_TOKEN_COUNT = 1                # 1 = off; 5 keeps tokens seen in >= 5 training descriptions

# ---- vectorizer (both models) ---------------------------------------------------
VECTORIZER = VectorizerSettings(
    kind="count",                  # "count" | "tfidf"
    ngram_range=(1, 2),
    unordered_bigrams=True,        # bidirectional bigrams: 'ball valve' == 'valve ball'
    min_df=3,                      # int = document count
    max_df=1.0,                    # float = proportion
    max_features=None,             # None = no cap
    stop_words=None,               # "english" also drops FRONT, BACK, BOTTOM, FILL ...
    binary=True,
    # tfidf only
    sublinear_tf=True, norm="l2", use_idf=True, smooth_idf=True,
)

# ---- Naive Bayes -----------------------------------------------------------------
NB = NBSettings(alpha=0.001, fit_prior=True, class_prior=None)

# ---- neural network (ignored for NB) --------------------------------------------
NN = dict(
    hidden_dims=(512, 256), dropout=0.2, batch_norm=True, feature_dropout=0.0,
    batch_size=512, epochs=100, lr=1e-3, weight_decay=1e-5, label_smoothing=0.0,
    level_loss_weights=None,       # e.g. (1, 1, 1, 2) to weight the deepest level
    sparse_grad=True,              # SparseAdam on the first layer; set False if torch objects
    lr_scheduler=True, scheduler_patience=3, scheduler_factor=0.5,
    early_stopping=True, patience=6,
    monitor="accuracy",            # "accuracy" (deepest level) | "val_loss"
    monitor_level=-1,
    checkpoint=True,               # best epoch kept on disk and restored
    prefer_gpu=True,               # CUDA if available, else CPU
    num_workers=0, eval_batch_size=4096, seed=42,
)

# ---- turning per-level probabilities into paths ----------------------------------
PATH_MODE = "independent"          # "independent" | "conditional"
NORMALISE = True                   # normalise path scores over valid paths
SCORE_BATCH_SIZE = 2000            # rows per path-scoring block (memory)
PREDICT_CHUNK_SIZE = 50000         # rows per prediction chunk (memory)

# ---- analysis and saving -----------------------------------------------------------
BUCKET_WIDTH = 0.05
N_QUANTILE_BUCKETS = None          # e.g. 20 for equal-sized buckets instead
REFERENCE_ACCURACY = 0.95          # dashed reference line on the charts; None = off
TOP_CONFUSIONS = 20
SAVE_MODEL_CONFIG = True
NOTES = ""

## 1 · Data

In [ ]:
cleaning_config, cleaner = io.load_cleaning(CONFIG_DIR, CLEANING_STAMP)
df = io.load_labelled(DATA_DIR, cleaning_config, ID_COL, blacklist_file=BLACKLIST_FILE)
lookup = hierarchy_lookup(df, LEVEL_COLUMNS, NAME_COLUMNS)

if TEST_SET_FILE:
    test_ids = io.load_id_list(f"{DATA_DIR}/{TEST_SET_FILE}", ID_COL)
    train_df, test_df = io.split_off_ids(df, test_ids, ID_COL)
    missing = len(test_ids - set(test_df[ID_COL]))
    print(f"{len(test_df):,} test rows; {missing:,} test IDs not in the labelled data "
          "(unlabelled, dropped by cleaning, a rare path, or blacklisted)")
else:
    from sklearn.model_selection import train_test_split
    train_df, test_df = train_test_split(df, test_size=TEST_FRAC, random_state=SEED)
    train_df, test_df = train_df.reset_index(drop=True), test_df.reset_index(drop=True)
print(f"{len(train_df):,} training rows, {len(test_df):,} test rows")

unseen = ~test_df[LEVEL_COLUMNS[-1]].isin(set(train_df[LEVEL_COLUMNS[-1]]))
if unseen.any():
    print(f"{int(unseen.sum()):,} test rows belong to a leaf category absent from training "
          "and cannot be predicted correctly")

## 2 · Train

In [ ]:
cfg = ModelConfig(
    model_type=MODEL_TYPE, text_col=TEXT_COL, id_col=ID_COL,
    level_columns=LEVEL_COLUMNS, name_columns=NAME_COLUMNS,
    min_token_count=MIN_TOKEN_COUNT, vectorizer=VECTORIZER, nb=NB, nn=dict(NN),
    path_mode=PATH_MODE, normalise=NORMALISE, score_batch_size=SCORE_BATCH_SIZE,
    predict_chunk_size=PREDICT_CHUNK_SIZE,
    cleaning_stamp=CLEANING_STAMP, blacklist_file=BLACKLIST_FILE, notes=NOTES,
)

val_df = None
if MODEL_TYPE == "nn" and NN_VAL_FRAC:
    train_df, val_df = validation_split(train_df, NN_VAL_FRAC, LEVEL_COLUMNS[-1], SEED)

EVAL_DIR = f"{OUTPUT_DIR}/eval_{STAMP}"
clf = CategoryClassifier(cfg).fit(train_df, val_df, checkpoint_dir=f"{EVAL_DIR}/checkpoint")

In [ ]:
if clf.history is not None:
    fig_curves = plotting.plot_training_curves(clf.history, LEVEL_COLUMNS)

## 3 · Predict the test set

In [ ]:
pred = clf.predict(test_df[TEXT_COL])
pred = pd.concat([test_df[[ID_COL, TEXT_COL] + LEVEL_COLUMNS].reset_index(drop=True), pred], axis=1)
SPLIT_COL = None
pred.head()

### Overall

In [ ]:
m = evaluation.metrics(pred, LEVEL_COLUMNS)
display(pd.Series(m, name="value").to_frame())
fig = plotting.plot_per_level_accuracy(m, LEVEL_COLUMNS)

### By confidence bucket

`L<d> Confidence` is the share of path probability under the predicted prefix down to
depth d. Each depth is bucketed on its own distribution; read each panel on its own.

In [ ]:
buckets = evaluation.bucket_accuracy(pred, LEVEL_COLUMNS, split_col=SPLIT_COL,
                                     bucket_width=BUCKET_WIDTH, n_quantile_buckets=N_QUANTILE_BUCKETS)
summaries = evaluation.bucket_summary(buckets)
summaries[len(LEVEL_COLUMNS)]

In [ ]:
figs = {
    "accuracy_by_bucket": plotting.plot_accuracy_by_bucket(summaries[len(LEVEL_COLUMNS)],
                                                           target_accuracy=REFERENCE_ACCURACY),
    "coverage": plotting.plot_coverage_curve(summaries[len(LEVEL_COLUMNS)], REFERENCE_ACCURACY),
    "accuracy_by_bucket_per_level": plotting.plot_accuracy_by_bucket_grid(
        summaries, LEVEL_COLUMNS, target_accuracy=REFERENCE_ACCURACY),
    "coverage_per_level": plotting.plot_coverage_curve_grid(
        summaries, LEVEL_COLUMNS, target_accuracy=REFERENCE_ACCURACY),
}

### Most frequent confusions

In [ ]:
confusions = evaluation.top_confusions(pred, LEVEL_COLUMNS[-1], n=TOP_CONFUSIONS,
                                       lookup=lookup, name_col=NAME_COLUMNS[-1])
figs["confusions"] = plotting.plot_top_confusions(confusions, LEVEL_COLUMNS[-1])
confusions

In [ ]:
unreadable = pred[~pred["readable"].astype(bool)]
print(f"{len(unreadable):,} of {len(pred):,} rows had no feature the model knows "
      f"(their predictions come from class frequencies alone)")
unreadable[[ID_COL, TEXT_COL]].head(10)

## 4 · Save

In [ ]:
import shutil
os.makedirs(EVAL_DIR, exist_ok=True)
pred.to_csv(f"{EVAL_DIR}/test_predictions.csv", index=False)
io.save_json({"stamp": STAMP, "cleaning_stamp": CLEANING_STAMP, "test_set_file": TEST_SET_FILE,
              "n_train": len(train_df), "n_val": 0 if val_df is None else len(val_df),
              "n_test": len(test_df), "metrics": m}, f"{EVAL_DIR}/metrics.json")
for name, f in figs.items():
    f.savefig(f"{EVAL_DIR}/{name}.png", dpi=120, bbox_inches="tight")
if clf.history is not None:
    clf.history.to_csv(f"{EVAL_DIR}/history.csv", index=False)
    fig_curves.savefig(f"{EVAL_DIR}/training_curves.png", dpi=120, bbox_inches="tight")
shutil.rmtree(f"{EVAL_DIR}/checkpoint", ignore_errors=True)
cfg.save(f"{EVAL_DIR}/model_config.json")

if SAVE_MODEL_CONFIG:
    cfg.source = f"eval_{STAMP}"
    cfg.save(f"{CONFIG_DIR}/model_{STAMP}.json")
    print(f"\nMODEL_CONFIG_FILE = \"model_{STAMP}.json\"   <- use this in notebook 04")